In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
import joblib
import warnings
warnings.filterwarnings('ignore')

class FraudDataPreprocessor:
    """
    A comprehensive data preprocessor for credit card fraud detection.
    """
    
    def __init__(self):
        self.scaler = None
        self.feature_columns = None
        self.target_column = 'Class'
        
    def load_data(self):
        """Load the credit card fraud dataset."""
        df = pd.read_csv('../data/raw/creditcard.csv')
        return df
    
    def engineer_features(self, df):

        
        df_enhanced = df.copy()
        
        # Time-based features
        df_enhanced['Hour'] = (df_enhanced['Time'] % (24 * 3600)) // 3600
        df_enhanced['Day'] = df_enhanced['Time'] // (24 * 3600)
        
        # Amount-based features
        df_enhanced['Amount_log'] = np.log(df_enhanced['Amount'] + 1)
        df_enhanced['Amount_sqrt'] = np.sqrt(df_enhanced['Amount'])
        
        # Interaction features with top discriminative features
        top_features = ['V14', 'V4', 'V11', 'V12', 'V10']
        for i, feat1 in enumerate(top_features):
            for feat2 in top_features[i+1:]:
                df_enhanced[f'{feat1}_{feat2}_interaction'] = df_enhanced[feat1] * df_enhanced[feat2]
        
        
        return df_enhanced
        
    def prepare_data_splits(self, df, test_size=0.2, val_size=0.2, random_state=42):
    
        # Separate features and target
        X = df.drop([self.target_column], axis=1)
        y = df[self.target_column]
        
        # Store feature columns
        self.feature_columns = X.columns.tolist()
        
        # First split: train+val vs test
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        
        # Second split: train vs val
        adjusted_val_size = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=adjusted_val_size, random_state=random_state, stratify=y_temp
        )
        
        return X_train, X_val, X_test, y_train, y_val, y_test

    def scale_features(self, X_train, X_val, X_test, scaler_type='standard'):
        
        if scaler_type == 'standard':
            self.scaler = StandardScaler()
        elif scaler_type == 'robust':
            self.scaler = RobustScaler()
        else:
            raise ValueError("scaler_type must be 'standard' or 'robust'")
        
        # Fit on training data only
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Convert back to DataFrames
        X_train_scaled = pd.DataFrame(X_train_scaled, columns=self.feature_columns, index=X_train.index)
        X_val_scaled = pd.DataFrame(X_val_scaled, columns=self.feature_columns, index=X_val.index)
        X_test_scaled = pd.DataFrame(X_test_scaled, columns=self.feature_columns, index=X_test.index)
        
        return X_train_scaled, X_val_scaled, X_test_scaled